ATTENTION: For now this code has successfully run ONLY if you attach the notebook to a Starter Warehouse SQL in Databricks and does not work accross environments. It works only in prod environment.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW MaxBussDate
AS SELECT *
FROM
(
select max(a.businessdate) as MaxDate from (

(select distinct businessdate from `p-cdh`.derivative.dexposure) a 
inner join (select distinct businessdate from `p-cdh`.derivative.DAuctionRealEstate) b on a.businessdate = b.businessdate
inner join (select distinct businessdate from `p-cdh`.derivative.dparty) c on b.businessdate = c.businessdate
inner join (select distinct businessdate from `p-cdh`.primary.exposure) d on c.businessdate = d.businessdate
inner join (select distinct businessdate from `p-cdh`.derivative.danalyticsmetrics) e on d.businessdate = e.businessdate
inner join (select distinct businessdate from `p-cdh`.primary.settlementexposure) f on e.businessdate = f.businessdate
inner join (select distinct businessdate from `p-cdh`.primary.restructuring ) g on f.businessdate = g.businessdate
inner join (select distinct businessdate from `p-cdh`.derivative.dunderlaw) h on g.businessdate = h.businessdate
inner join (select distinct businessdate from `p-cdh`.derivative.dlegalprocessstatusagreement) i on h.businessdate = i.businessdate
inner join (select distinct businessdate from `p-cdh`.primary.legalprocessstage) j on i.businessdate = j.businessdate
inner join (select distinct businessdate from `p-cdh`.primary.firmstructure) k on j.businessdate = k.businessdate
inner join (select distinct businessdate from `p-cdh`.primary.assignment) l on k.businessdate = l.businessdate) )


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW auct
AS SELECT *
FROM
(select distinct BusinessDate,OriginDebtorTaxRegistrationNumber, DebtorTaxRegistrationNumber, DebtorDescription,
ROW_NUMBER() OVER (PARTITION BY BusinessDate,OriginDebtorTaxRegistrationNumber, DebtorTaxRegistrationNumber ORDER BY DebtorDescription desc) AS Rank 
from `p-cdh`.derivative.DAuctionRealEstate
where BusinessDate = (select MaxDate from MaxBussDate) and DebtorTaxRegistrationNumber not in ('000000000','0000000000'))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW d360
AS SELECT *
FROM
(
select distinct IncomingAuctionKey,IncomingAuctionObjKey,SourceSystem,MatchingPercentage,UIMatchingAuctionStatus,SourceRealEstateKey,
ROW_NUMBER() OVER (PARTITION BY IncomingAuctionKey,IncomingRealEstateKey ORDER BY SourceSystem DESC) AS Ranking
from (
select * from `p-cdh`.derivative.dauction360matching a
  join (
    select
      AuctionMatching360ID,
      max(IngestionTimestamp) as IngestionTimestamp
    from
      `p-cdh`.derivative.dauction360matching
    group by AuctionMatching360ID
  ) b on a.AuctionMatching360ID = b.AuctionMatching360ID
  and a.IngestionTimestamp = b.IngestionTimestamp and (--a.MatchingPercentage=1 OR 
  a.UIMatchingAuctionStatus = 'validated-matched')
  ) )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW d360_2
AS SELECT *
FROM
(
select distinct IncomingAuctionKey,IncomingAuctionObjKey,IncomingisLastAuction
from (
select * from `p-cdh`.derivative.dauction360matching a
  join (
    select
      AuctionMatching360ID,
      max(IngestionTimestamp) as IngestionTimestamp
    from
      `p-cdh`.derivative.dauction360matching
    group by AuctionMatching360ID
  ) b on a.AuctionMatching360ID = b.AuctionMatching360ID
  and a.IngestionTimestamp = b.IngestionTimestamp ) where IncomingisLastAuction = 'yes')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW prog
AS SELECT *
FROM
(
select distinct 
bb.c_auction_id, bb.GroupFirstAuction,
ROW_NUMBER() OVER (PARTITION BY bb.GroupFirstAuction ORDER BY bb.auctiondate) as AuctionProgramNo, 
case when (ROW_NUMBER() OVER (PARTITION BY bb.GroupFirstAuction ORDER BY bb.auctiondate DESC)) = 1 then 'yes' else null end as GroupLastAuction
from (
select 
bean.c_auction_id, bean.auctiondate, bean.GroupFirstAuction
from (
select distinct 
g.c_auction_id, g.AuctionDate,g.GroupFirstAuction,g.RealEstateGroupCode,
ROW_NUMBER() OVER (PARTITION BY g.c_auction_id ORDER BY g.RealEstateGroupCode) AS Rank2
from (
select distinct
cast(fa.c_auction_id as string) as GroupFirstAuction,
fa.RealEstateGroupCode, fa.c_auction_id as FirstAuction,tia.c_auction_id, tia.AuctionDate
from `p-cdh-delivery`.tia.tia tia

inner join (
select RealEstateGroupCode, c_auction_id, AuctionDate,
ROW_NUMBER() OVER (PARTITION BY RealEstateGroupCode ORDER BY auctiondate) AS Rank1
from (select distinct RealEstateGroupCode, c_auction_id, AuctionDate
from  `p-cdh-delivery`.tia.tia)) fa 

on tia.RealEstateGroupCode = fa.RealEstateGroupCode and fa.Rank1 = 1 ) g ) bean 
where bean.Rank2 = 1 ) bb)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tia
AS SELECT *
FROM
(
select distinct bi.*,
case when bi.Initiator_TIA in (
'Alpha Bank',
'Cepal Undefined Initiator',
'Cosmos',
'Galaxy II',
'Gemini',
'GXR One Holding',
'Hoist',
'Orion') then 'yes' else 'no' end as TiaRetailFlag,
case when prog.GroupFirstAuction  is null then bi.OriginAuctionKey          else prog.GroupFirstAuction       end as GroupFirstAuction_f,
case when prog.GroupFirstAuction  is null then bi.count_auctions            else prog.AuctionProgramNo        end as AuctionProgramNo_f,
case when prog.GroupFirstAuction  is null then d.IncomingisLastAuction      else prog.GroupLastAuction        end as GroupLastAuction_f
from `p-cdh-delivery`.tia.tia bi
left join 
d360_2 d
on bi.c_auction_id = d.IncomingAuctionKey and bi.AuctionObJID = d.IncomingAuctionObjKey
left join
prog
on bi.c_auction_id = prog.c_auction_id)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW LastProp
AS SELECT *
FROM
(
select RealEstateGroupCode, max(AuctionProgramNo_f) as MaxAuctionProgramNo_f from tia
where RealEstateGroupCode is not null group by RealEstateGroupCode)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Miss
AS SELECT *
FROM
(
select dauct.OriginAuctionKey as GroupFirstAuction_f,
ROW_NUMBER() OVER (PARTITION BY dauct.OriginAuctionKey ORDER BY dauct.Conductiondate) as AuctionProgramNo_f, 
case when (ROW_NUMBER() OVER (PARTITION BY dauct.OriginAuctionKey ORDER BY dauct.Conductiondate DESC)) = 1 then 'yes' else null end as GroupLastAuction_ff
,dauct.AuctionKey as c_auction_id
,dauct.AuctionKey as UniqueCode

, null as FinalPrice
, null as KAEK
, null as RealEstateGroupCode
, null as AuctionObJID
, dauct.OriginAuctionKey as OriginAuctionKey
, null as count_auctions
,CASE
    WHEN dauct.Conductiondate REGEXP '^[0-9]+\\.[0-9]+$' THEN NULL  -- Handle numeric strings
    ELSE CAST(dauct.Conductiondate AS string)
    END AS AuctionDate
,dauct.DebtorTaxRegistrationNumber as DebtorVatNumber
,dauct.DebtorTaxRegistrationNumber as OwnerTaxRegistrationNumber
,dauct.DebtorDescription as OwnerName
, null as DebtorDescr
,dauct.SourceSystem as Source
, null as AuctioneerName
, null as AuctioneerPhone
, null as AuctioneerEmail
, null as Prefecture
,dauct.Servicer as BasicDataServicer
, null as IncomingPortfolioCategory
, null as PostCode
, null as CategoryGR
, null as SubCategoryGR

,dauct.IsHastenerServicer as IncomingIsHastenerServicer
,dauct.Initiator as Initiator_TIA
,dauct.InitiatorCepalAB as Initiator_Cepal_AB
,dauct.Hastener  as Initiator_Cepal_others
,dauct.AuctionStatusCode as Result_blended
,CASE
    WHEN dauct.Publisheddate REGEXP '^[0-9]+\\.[0-9]+$' THEN NULL  -- Handle numeric strings
    ELSE CAST(dauct.Publisheddate AS string)
  END AS Publisheddate_f
,dauct.FirstOfferPrice as FirstOfferPrice
, null as cepal_key_final_thesis
, null as SourceRealEstateKey
, null as MatchingPercentage
, null as sourcesystem
, null as UIMatchingAuctionStatus
, null as IncomingisLastAuction
,case when dauct.Initiator in (
'Alpha Bank',
'Cepal Undefined Initiator',
'Cosmos',
'Galaxy II',
'Gemini',
'GXR One Holding',
'Hoist',
'Orion') then 'yes' else 'no' end as TiaRetailFlag

from `p-cdh`.derivative.dauction dauct
left join
(select distinct c_auction_id from `p-cdh-delivery`.tia.tia) tia
on dauct.AuctionKey = tia.c_auction_id
where dauct.referencedate = (select max(referencedate) from `p-cdh`.derivative.dauction)
and tia.c_auction_id is null
and dauct.AuctionKey <> '0')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dexp
AS SELECT *
FROM
(
select *,
case when PortfolioName in ('Core Retail','Core Wholesale') then AgreementKey
when CHARINDEX('-',AgreementKey) = 0 then RIGHT(AgreementKey,LEN(AgreementKey)-3) 
else SUBSTRING(AgreementKey, 4,CHARINDEX('-',AgreementKey) -4) 
END as AgreementNumberNoPrefix
,case when PortfolioName in ('Core Retail','Core Wholesale') then ExposureKey else RIGHT(ExposureKey,LEN(ExposureKey)-3) END as ExposureKeyNoPrefix
,case when spv = 'GEMINI CORE SECURITISATION DAC' then 'GEMINI CORE SECURITISATION DAC' else PortfolioName end as Portfolio
from `p-cdh`.derivative.dexposure where IsActiveFlag = 'true' and businessdate =  (select MaxDate from MaxBussDate) )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW expDexp
AS SELECT *
FROM
(
select               exp.PartyTaxRegistrationNumber               as ExpTaxNumber
                    ,exp.cdi                                      as ExpCDI
                    ,exp.PartyRelationshipType                    as RelationType
                    ,exp.SourceSystem                             as ExpSourceSystem
                    ,exp.PartyKey                                 as ExpPartyKey
,dexp.*
from (select * from `p-cdh`.primary.exposure where businessdate =  (select MaxDate from MaxBussDate) ) exp 
inner join 
dexp 
on  exp.ExposureKey =dexp.ExposureKey
and exp.BusinessDate=dexp.BusinessDate 
and exp.SourceSystem=dexp.SourceSystem)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Exprt
AS SELECT *
FROM
(
  select distinct mm.AgreementKey, mm.AgreementNumberNoPrefix, mm.PortfolioName, mm.PrevPortfolio, mm.isactiveflag 
	from (
	select distinct m.AgreementNumberNoPrefix, m.AgreementKey, m.PortfolioName, m.isactiveflag,
	LAG(portfolioname) OVER (PARTITION BY AgreementNumberNoPrefix ORDER BY isactiveflag) AS PrevPortfolio
	from (
	select distinct PortfolioName, AgreementKey, isactiveflag,
	case when PortfolioName in ('Core Retail','Core Wholesale') then AgreementKey
    when sourcesystem <> 'qca' and CHARINDEX('-',AgreementKey) = 0  then RIGHT(AgreementKey,LEN(AgreementKey)-3) 
    when sourcesystem <> 'qca' and CHARINDEX('-',AgreementKey) <> 0 then SUBSTRING(AgreementKey, 4,CHARINDEX('-',AgreementKey) -4) 
		else AgreementKey
    END as AgreementNumberNoPrefix			
	from `p-cdh`.primary.exposure
	) m
	) mm where mm.portfolioname <> mm.PrevPortfolio and mm.isactiveflag = 'true')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW lpsAGR
AS SELECT *
FROM
(
select *,ROW_NUMBER() OVER (PARTITION BY businessdate, AgreementKey ORDER BY OperationalLegalStatusLevel3Code DESC) AS Ranking
from `p-cdh`.derivative.dlegalprocessstatusagreement
where BusinessDate =  (select MaxDate from MaxBussDate) )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW LoanInter
AS SELECT *
FROM
(
 -- τιμή owner  ->  τότε το πεδίο TaxNumber είναι: ο ιδιοκτητης του πλειστηριασμου αλλα όχι ο πρωτοφειλετης της σύμβασης που έγινε η κατασχεση.
 -- τιμή owner/main debtor   ->  τότε το πεδίο TaxNumber είναι: ο ιδιοκτήτης και ο πρωτοφειλέτης της συμβασης που εγινε η κατασχεση.
 -- τιμή main debtor  ->  τότε το πεδίο TaxNumber είναι: ο πρωτοφειλέτης της σύμβασης που έγινε η κατασχεση αλλά όχι ο ιδιοκτήτης. 

select distinct 
      dexp.BusinessDate
     ,exp.PartyTaxRegistrationNumber as PropertyOwnerTaxNumber
     ,exp.PartyTaxRegistrationNumber as TaxNumber
     ,case when max(case when exp.PartyRelationshipType in ('Main Beneficiary','Primary Owner') and dexp.isActiveFlag = 'true' and
      lpsAGR.LegalProcessStatusLevel1 = 'Αναγκαστική Εκτέλεση' then 1 else 0 end) = 1 then 'owner/main debtor' else 'owner' end as Ofeiles
from
(select * from `p-cdh`.primary.exposure where BusinessDate =  (select MaxDate from MaxBussDate) ) exp 
inner join 
dexp
on  exp.ExposureKey =dexp.ExposureKey
and exp.BusinessDate=dexp.BusinessDate 
and exp.SourceSystem=dexp.SourceSystem
left join
lpsAGR
on  dexp.businessdate = lpsAGR.businessdate   
and dexp.AgreementKey = lpsAGR.AgreementKey and lpsAGR.ranking = 1
where dexp.isActiveFlag = 'true'
group by dexp.BusinessDate, exp.PartyTaxRegistrationNumber , exp.PartyTaxRegistrationNumber
UNION
select distinct 
      dexp.BusinessDate
     ,exp1.PartyTaxRegistrationNumber as PropertyOwnerTaxNumber
     ,exp2.PartyTaxRegistrationNumber as TaxNumber
     ,'main debtor' as Ofeiles
from
(select * from `p-cdh`.primary.exposure where BusinessDate =  (select MaxDate from MaxBussDate) ) exp1
inner join
`p-cdh`.primary.exposure exp2
on  exp1.ExposureKey =exp2.ExposureKey
and exp1.BusinessDate=exp2.BusinessDate 
and exp1.SourceSystem=exp2.SourceSystem
inner join 
dexp
on  exp1.ExposureKey =dexp.ExposureKey
and exp1.BusinessDate=dexp.BusinessDate 
and exp1.SourceSystem=dexp.SourceSystem
left join
lpsAGR
on  dexp.businessdate = lpsAGR.businessdate   
and dexp.AgreementKey = lpsAGR.AgreementKey and lpsAGR.ranking = 1
where dexp.isActiveFlag = 'true' and exp1.PartyTaxRegistrationNumber <> exp2.PartyTaxRegistrationNumber
and exp2.PartyRelationshipType in ('Main Beneficiary','Primary Owner') and lpsAGR.LegalProcessStatusLevel1 = 'Αναγκαστική Εκτέλεση')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW crv
AS SELECT *
FROM
(
select businessdate, AgreementKey, sum(CREVBaseTLC) as SumCrevBase from `p-cdh`.derivative.danalyticsmetrics where BusinessDate =  (select MaxDate from MaxBussDate) 
group by businessdate,AgreementKey)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sett
AS SELECT *
FROM
(
select  ExposureKey,BusinessDate,SourceSystem,SettlementType,SettlementStatus,IsActiveFlag,Bucket,SubmissionDate,
NextInstalmentdate,DebtAmount,SettlementAmount,LastUpdatedDate,ImplementationDate,SettlementKey,
row_number() over (partition by ExposureKey,BusinessDate order by SubmissionDate desc) as rnk
from  `p-cdh`.primary.settlementexposure
where businessdate =  (select MaxDate from MaxBussDate) )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW restr
AS SELECT *
FROM
(
select businessdate, SourceSystem, OldAgreementKey, NewAgreementKey, OldExposureKey, NewExposureKey,RestructuringType,
RestructuringDate, ROW_NUMBER() OVER (PARTITION BY NewExposureKey,BusinessDate ORDER BY RestructuringDate DESC) AS Ranking  
from `p-cdh`.primary.restructuring
where businessdate =  (select MaxDate from MaxBussDate) )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW OCW
AS SELECT *
FROM
(
select CDI,BusinessDate,LawSystem,UnderLawType,ApplicationStatus,ApplicationStatusDate,CourtDate,
ROW_NUMBER() OVER (PARTITION BY CDI ORDER BY businessdate desc, ApplicationStatusDate DESC) AS Ranking  
from `p-cdh`.derivative.dunderlaw
where LawSystem in ('Εξωδικαστικός μηχανισμός','Βεβαίωση ευάλωτου') )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW L3869
AS SELECT *
FROM
(
select  distinct CDI,businessdate,UnderLawType,InLawFlag,ApplicationStatus,LawStatus,CalcLawStatus,ApplicationStatusDate,CourtDate
FROM    `p-cdh`.derivative.dunderlaw 
WHERE   UnderLawType='N3869' and businessdate =  (select MaxDate from MaxBussDate) )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW prcs_KSX
AS SELECT *
FROM
(
             select distinct
                  dexp.agreementkey
                , dexp.sourcesystem                    
                , dexp.BusinessDate
                , legpro.LegalProcessDefinitionCode as LegalProcessKSX
                , legpro.LegalStageDefinition as  LegalStepStageNameKSX    
                , legpro.LegalProcessStatus as LegalProcessStatusKSX
                , legpro.ExternalCompanyAssignmentKey
                , firm.EmployeeFullName as ConfiscationCurrentAgency
                , firm.EmployeeCategory 
                , legpro.LegalProcessStartDate as EntryProcessDateKSX
                , ROW_NUMBER() OVER (PARTITION BY dexp.Businessdate, dexp.AgreementKey ORDER BY legpro.LegalProcessStartDate desc,legpro.LegalProcessStatus) AS Rank

    from        dexp
    INNER join  `p-cdh`.primary.legalprocessstage legpro
    on          dexp.CaseCode=legpro.CaseCode
    and         regexp_replace(regexp_extract(dexp.ExposureKey, '\\d+', 0), '^0+', '') = regexp_replace(regexp_extract(legpro.ExposureKey, '\\d+', 0), '^0+', '') 
    and         dexp.BusinessDate=legpro.BusinessDate
    LEFT JOIN  
        (select distinct businessdate, SourceSystem, EmployeePartyKey, EmployeeCategory, EmployeeFullName from `p-cdh`.primary.firmstructure 
    where EmployeeCategory='External Company' and EmployeeFullName is not null and businessdate =  (select MaxDate from MaxBussDate) ) firm
    on          cast(legpro.ExternalCompanyAssignmentKey as string)=cast(firm.EmployeePartyKey as string)
    and         legpro.BusinessDate=firm.BusinessDate
    and         legpro.SourceSystem=firm.SourceSystem
    WHERE       legpro.LegalStageDefinition='Προς Επιβολή Κατάσχεσης')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW prcs_DP
AS SELECT *
FROM
(   select   distinct    
                  dexp.agreementkey
                , dexp.sourcesystem                    
                , dexp.BusinessDate
                , legpro.LegalProcessDefinitionCode as LegalProcessDP
                , legpro.LegalStageDefinition as  LegalStepStageNameDP    
                , legpro.LegalProcessStatus as LegalProcessStatusDP
                , legpro.ExternalCompanyAssignmentKey
                , firm.EmployeeFullName as PaymentOrderCurrentAgency 
                , firm.EmployeeCategory 
                , legpro.LegalProcessStartDate as EntryProcessDateDP
                , ROW_NUMBER() OVER (PARTITION BY dexp.Businessdate, dexp.AgreementKey ORDER BY legpro.LegalProcessStartDate desc,legpro.LegalProcessStatus) AS Rank

    from        dexp
    INNER join   `p-cdh`.primary.legalprocessstage legpro
    on          dexp.CaseCode=legpro.CaseCode
    and         regexp_replace(regexp_extract(dexp.ExposureKey, '\\d+', 0), '^0+', '') = regexp_replace(regexp_extract(legpro.ExposureKey, '\\d+', 0), '^0+', '') 
    and         dexp.BusinessDate=legpro.BusinessDate
    LEFT JOIN  
        (select distinct businessdate, SourceSystem, EmployeePartyKey, EmployeeCategory, EmployeeFullName from `p-cdh`.primary.firmstructure 
    where EmployeeCategory='External Company' and EmployeeFullName is not null and businessdate =  (select MaxDate from MaxBussDate) ) firm
    on          cast(legpro.ExternalCompanyAssignmentKey as string)=cast(firm.EmployeePartyKey as string)
    and         legpro.BusinessDate=firm.BusinessDate
    and         legpro.SourceSystem=firm.SourceSystem
    where         (legpro.LegalStageDefinition = 'Προς Σύνταξη /Κατάθεση/Έκδοση Δ/Π' or legpro.LegalStageDefinition like 'Α'''' επίδοση'))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW prcs_AG
AS SELECT *
FROM
(
     select   distinct    
                  dexp.agreementkey
                , dexp.sourcesystem                    
                , dexp.BusinessDate
                , legpro.LegalProcessDefinitionCode as LegalProcessAG
                , legpro.LegalStageDefinition as  LegalStepStageNameAG    
                , legpro.LegalProcessStatus as LegalProcessStatusAG
                , legpro.ExternalCompanyAssignmentKey
                , firm.EmployeeFullName as LawsuitCurrentAgency 
                , firm.EmployeeCategory 
                , legpro.LegalProcessStartDate as EntryProcessDateAG
                , ROW_NUMBER() OVER (PARTITION BY dexp.Businessdate, dexp.AgreementKey ORDER BY legpro.LegalProcessStartDate) AS Rank

    from        dexp
    INNER join   `p-cdh`.primary.legalprocessstage legpro
    on          dexp.CaseCode=legpro.CaseCode
    and         dexp.BusinessDate=legpro.BusinessDate
    LEFT JOIN  
        (select distinct businessdate, EmployeePartyKey, EmployeeCategory, EmployeeFullName from `p-cdh`.primary.firmstructure 
    where EmployeeCategory='External Company' and EmployeeFullName is not null and businessdate =  (select MaxDate from MaxBussDate) ) firm
    on          cast(legpro.ExternalCompanyAssignmentKey as string)=cast(firm.EmployeePartyKey as string)
    and         legpro.BusinessDate=firm.BusinessDate
    where         (legpro.LegalStageDefinition like 'Προς Κατάθεση Αγωγής%'))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW prcs_PROS
AS SELECT *
FROM
(
     select   distinct    
                  dexp.agreementkey
                , dexp.sourcesystem                    
                , dexp.BusinessDate
                , legpro.LegalProcessDefinitionCode as LegalProcessPROS
                , legpro.LegalStageDefinition as  LegalStepStageNamePROS    
                , legpro.LegalProcessStatus as LegalProcessStatusPROS
                , legpro.ExternalCompanyAssignmentKey
                , firm.EmployeeFullName as PrenotationCurrentAgency
                , firm.EmployeeCategory 
                , legpro.LegalProcessStartDate as EntryProcessDatePROS
                , ROW_NUMBER() OVER (PARTITION BY dexp.Businessdate, dexp.AgreementKey ORDER BY legpro.LegalProcessStartDate desc,legpro.LegalProcessStatus) AS Rank

    from        dexp
    INNER join   `p-cdh`.primary.legalprocessstage legpro
    on          dexp.CaseCode=legpro.CaseCode
    and         regexp_replace(regexp_extract(dexp.ExposureKey, '\\d+', 0), '^0+', '') = regexp_replace(regexp_extract(legpro.ExposureKey, '\\d+', 0), '^0+', '') 
    and         dexp.BusinessDate=legpro.BusinessDate
    LEFT JOIN  
        (select distinct businessdate, SourceSystem, EmployeePartyKey, EmployeeCategory, EmployeeFullName from `p-cdh`.primary.firmstructure 
    where EmployeeCategory='External Company' and EmployeeFullName is not null and businessdate =  (select MaxDate from MaxBussDate) ) firm
    on          cast(legpro.ExternalCompanyAssignmentKey as string)=cast(firm.EmployeePartyKey as string)
    and         legpro.BusinessDate=firm.BusinessDate
    and         legpro.SourceSystem=firm.SourceSystem
    where       legpro.LegalStageDefinition like 'Προς Εγγραφή Βάρους' OR legpro.LegalStageDefinition like 'Εγγραφή αναγκαστικής προσημείωσης')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW strm
AS SELECT *
FROM
(
select  assign.BusinessDate,assign.ExposureKey,assign.StreamType, max(firm.EmployeeFullName) as StreamAgency
               from        `p-cdh`.primary.assignment assign
                LEFT JOIN   `p-cdh`.primary.firmstructure firm
    on          cast(assign.externalCompanyPartyKey as string)=cast(firm.EmployeePartyKey as string)
    and         assign.sourceSystem=firm.SourceSystem
    and         assign.businessDate=firm.BusinessDate
    and         firm.employeeCategory='External Company'
    where       assign.businessDate =  (select MaxDate from MaxBussDate) 
                group by assign.BusinessDate,assign.ExposureKey,assign.AgreementKey,assign.StreamType)

In [0]:
%sql

CREATE OR REPLACE TABLE `p-cdh-delivery`.`tia`.auctionmonitoring

AS SELECT * FROM (

select 

BusinessDate
,CaseUniqueID
,case when AuctionFlag = '1.HMWN' and (CurrentInitiatorPortfolio = RankInitiator1) then '1.HMWN'
     when AuctionFlag in ('1.HMWN','2.TRITOU')                                     then '2.TRITOU'
     else AuctionFlag  end as AuctionsFlagf
,CurrentInitiatorPortfolio
,c_auction_id
,UniqueCode
,GroupFirstAuction_f as GroupFirstAuction 
,AuctionProgramNo_f  as AuctionProgramNo 
,case when Result_blended in ('eAuction - Ολοκληρωμένος - Κατακύρωση') then 'yes' else GroupLastAuction_ff end as GroupLastAuction
,AuctionObJID
,AuctionDate
,DebtorVatNumber
,OwnerTaxRegistrationNumber
,OwnerName
,Source
,AuctioneerName
,AuctioneerPhone
,AuctioneerEmail
,Prefecture
,BasicDataServicer
,IncomingIsHastenerServicer
,Initiator_TIA
,Initiator_Cepal_AB
,Initiator_Cepal_others
,Result_blended
,Publisheddate
,FirstOfferPrice
,SourceRealEstateKey
,MatchingPercentage
,sourcesystem
,Ofeiles
,expTaxNumber as TaxNumber
,PartyName
,PartyType
,expCDI as CDI
,RelationType
,AgreementNumber
,AgreementNumberNoPrefix
,AccountNumber
,AccountNumberNoPrefix
,Portfolio as PortfolioName
,PrevPortfolio
,AssetSegment
,matchedspv
,PrimaryDebtorCDI
,PrimaryDebtorTaxNumber
,LoanStatus
,IsActiveFlag
,Bucketf as Bucket
,DaysPastDueWithMateriality
,GBV
,OnBalance
,OffBalance
,TLC
,SumCrevBase
,SettlementType
,SettlementStatus
,SettlementActive
,SettlementBucket
,RestructureProduct
,RestructuringType as RestructureType 
,RestructuringDate
,LawType
,ApplicationStatus
,ApplicationStatusDate
,CourtDate
,LawType_3869
,InLawFlag_3869
,LawStatus_3869
,CourtDate_3869
,appstatus_3869
,appstatusdate_3869
,AgreementLPSDescr
,AgreementLPSDescr2
,LegalProcessKSX
,LegalProcessStatusKSX
,ConfiscationCurrentAgency
,EntryProcessDateKSX
,LegalStepStageNameKSX
,LegalProcessDP
,LegalProcessStatusDP
,PaymentOrderCurrentAgency
,EntryProcessDateDP
,LegalStepStageNameDP
,LegalProcessAG
,LegalProcessStatusAG
,LawsuitCurrentAgency
,EntryProcessDateAG
,LegalStepStageNameAG
,LegalProcessPROS
,LegalProcessStatusPROS
,PrenotationCurrentAgency
,EntryProcessDatePROS
,LegalStepStageNamePROS
,StreamAgency
,StreamType




from
(
select distinct *,
FIRST_VALUE(CurrentInitiatorPortfolio) OVER (PARTITION BY AuctionFlag, c_auction_id ORDER BY RankInitiator2) as RankInitiator1

from
(
select distinct al2.*,
DENSE_RANK() OVER (PARTITION BY al2.AuctionFlag, al2.c_auction_id ORDER BY al2.LPSOrder asc, al2.RelationTypeOrder asc, al2.OfeilesOrder asc) as RankInitiator2

from (

select 
case 
    when al.Initiator_Cepal_AB = 'Yes' and al.Matchedspv = 'yes' then '1.HMWN'
    when al.RetailWholesale is not null then '2.TRITOU'
    when al.TiaRetailFlag = 'yes' then '3.RetailInitiatorWithNoRetailOfeiles'
    else '4.OutOfScope'
end as AuctionFlag

,case when al.matchedspv  = 'yes' then al.Portfolio
      else                             al.Portfolio end as CurrentInitiatorPortfolio
,case when al.Ofeiles = 'owner/main debtor' then '1.owner/main debtor' 
      when al.Ofeiles = 'main debtor'       then '2.main debtor' 
      when al.Ofeiles = 'owner'             then '3.owner' end as OfeilesOrder 
,case when al.RelationType  in ('Co-Owner','Other')      then 3
      when al.RelationType  = 'Primary Owner'            then 1
      when al.RelationType  = 'Guarantor'                then 2 end as RelationTypeOrder
,case when al.AgreementLPSDescr = 'Αναγκαστική Εκτέλεση' then 1 else 2 end as LPSOrder

,al.*



from (

select tia2.c_auction_id||(case when tia2.AuctionObJID is null then 'X' else tia2.AuctionObJID end)||(case when tia2.OwnerTaxRegistrationNumber is null then 'X' else tia2.OwnerTaxRegistrationNumber end)||(case when dexp.expTaxNumber is null then 'X' else dexp.expTaxNumber end)
||(case when dexp.expCDI is null then 'X' else dexp.expCDI end)||(case when dexp.AgreementKey is null then 'X' else dexp.AgreementKey end)
||(case when dexp.ExposureKey is null then 'X' else dexp.ExposureKey end) as CaseUniqueID,
tia2.AuctionObJID||tia2.OwnerTaxRegistrationNumber||dexp.expTaxNumber||dexp.expCDI||dexp.AgreementKey||dexp.ExposureKey as CaseUniqueID2,
case when exists (
     select 1 from `p-ds`.reference.initiator_mapping m
     where m.label = tia2.Initiator_TIA
       and m.Matchedspv = 'yes'
       and (m.Cepal_Portfolio_Name is null
            or m.Cepal_Portfolio_Name = dexp.Portfolio
            or m.Cepal_Portfolio_Name = Exprt.PrevPortfolio)
      and (m.LPS is null or m.LPS = lpsAGR.LegalProcessStatusLevel1)
) then 'yes' else null end as Matchedspv
            ,tia2.*
            ,expX.Ofeiles
            ,dexp.BusinessDate
            ,dexp.expTaxNumber
            ,party.PartyType
            ,party.FirstNameGR ||' '|| party.LastNameGR as PartyName
            ,dexp.expCDI
            ,dexp.RelationType
            ,dexp.AgreementKey as AgreementNumber
            ,dexp.AgreementNumberNoPrefix
            ,dexp.ExposureKey as AccountNumber
            ,dexp.ExposureKeyNoPrefix as AccountNumberNoPrefix
            ,dexp.Portfolio
            ,st.RetailWholesale
            ,Exprt.PrevPortfolio
            ,dexp.AssetSegment
            ,dexp.cdi as PrimaryDebtorCDI   
            ,dexp.PartyTaxRegistrationNumber as PrimaryDebtorTaxNumber
            ,dexp.ExposureStatus as LoanStatus
            ,dexp.IsActiveFlag
            ,case when dexp.Bucket is null and dexp.DaysPastDueWithMateriality = 0 then 0 else dexp.Bucket end as Bucketf
            ,dexp.DaysPastDueWithMateriality
            ,cast(dexp.GBV as decimal (18,2))                      as GBV 
            ,cast(dexp.OnBalance as decimal (18,2))                as OnBalance 
            ,cast(dexp.OffBalance as decimal (18,2))               as OffBalance 
            ,cast(dexp.OutstandingTotalBalance as decimal (18,2))  as TLC

            ,cast(crv.SumCrevBase as decimal (18,2))               as SumCrevBase
 
            ,sett.SettlementType
            ,sett.SettlementStatus
            ,sett.IsActiveFlag as SettlementActive
            ,sett.Bucket as SettlementBucket

            ,restr.restructuringtype as RestructureProduct
            ,case when restr.RestructuringDate is not null then 'Restructure' else null end as RestructuringType 
            ,restr.RestructuringDate

            ,ocw.LawSystem as LawType
             ,ocw.ApplicationStatus
             ,ocw.ApplicationStatusDate
             ,ocw.CourtDate

             ,L3869.UnderLawType           as LawType_3869
             ,L3869.InLawFlag              as InLawFlag_3869
             ,L3869.CalcLawStatus          as LawStatus_3869
             ,L3869.CourtDate              as CourtDate_3869
             ,L3869.ApplicationStatus      as appstatus_3869
             ,L3869.ApplicationStatusDate  as appstatusdate_3869

            ,lpsAGR.LegalProcessStatusLevel1     as AgreementLPSDescr
            ,lpsAGR.OperationalLegalStatusLevel3 as AgreementLPSDescr2

            ,prcs_KSX.LegalProcessKSX 
            ,prcs_KSX.LegalProcessStatusKSX 
            ,prcs_KSX.ConfiscationCurrentAgency
            ,prcs_KSX.EntryProcessDateKSX
            ,prcs_KSX.LegalStepStageNameKSX

            ,prcs_DP.LegalProcessDP
            ,prcs_DP.LegalProcessStatusDP 
            ,prcs_DP.PaymentOrderCurrentAgency
            ,prcs_DP.EntryProcessDateDP
            ,prcs_DP.LegalStepStageNameDP 

            ,prcs_AG.LegalProcessAG
            ,prcs_AG.LegalProcessStatusAG
            ,prcs_AG.LawsuitCurrentAgency 
            ,prcs_AG.EntryProcessDateAG 
            ,prcs_AG.LegalStepStageNameAG 

            ,prcs_PROS.LegalProcessPROS
            ,prcs_PROS.LegalProcessStatusPROS
            ,prcs_PROS.PrenotationCurrentAgency
            ,prcs_PROS.EntryProcessDatePROS
            ,prcs_PROS.LegalStepStageNamePROS

            ,strm.StreamAgency
            ,strm.StreamType

FROM
((select distinct       
tia.GroupFirstAuction_f,tia.AuctionProgramNo_f,
case when coalesce(p.MaxAuctionProgramNo_f,null,0) = tia.AuctionProgramNo_f then 'yes' else tia.GroupLastAuction_f end as GroupLastAuction_ff,

tia.c_auction_id, tia.UniqueCode, tia.FinalPrice, tia.KAEK, tia.RealEstateGroupCode,tia.AuctionObJID, tia.OriginAuctionKey,tia.count_auctions,
tia.AuctionDate, tia.DebtorVatNumber, auct.DebtorTaxRegistrationNumber as OwnerTaxRegistrationNumber, auct.DebtorDescription as OwnerName, tia.DebtorDescr,
tia.Source, tia.AuctioneerName,tia.AuctioneerPhone,tia.AuctioneerEmail, tia.Prefecture,tia.BasicDataServicer,
tia.IncomingPortfolioCategory, tia.PostCode, tia.CategoryGR, tia.SubCategoryGR,
tia.IncomingIsHastenerServicer, tia.Initiator_TIA, tia.Initiator_Cepal_AB, tia.Initiator_Cepal_others, 
tia.Result_blended, 
tia.c_Published_date as Publisheddate, tia.FirstOfferPrice, tia.cepal_key_final_thesis,
d360.SourceRealEstateKey, d360.MatchingPercentage, d360.sourcesystem, d360.UIMatchingAuctionStatus,d360_2.IncomingisLastAuction,
tia.TiaRetailFlag
from 
tia tia
left join
LastProp p on tia.RealEstateGroupCode = p.RealEstateGroupCode
left join
auct
on tia.DebtorVatNumber = auct.OriginDebtorTaxRegistrationNumber
and auct.Rank = 1
left join
d360
on tia.c_auction_id = d360.IncomingAuctionKey  and tia.AuctionObJID = d360.IncomingAuctionObjKey and d360.Ranking = 1
left join
d360_2
on tia.c_auction_id = d360_2.IncomingAuctionKey and tia.AuctionObJID = d360_2.IncomingAuctionObjKey 
left join
(select distinct 
bi.GroupFirstAuction_f,bi.AuctionProgramNo_f,bi.AuctionDate as FirstProframAuctionDate
from tia bi
where bi.AuctionProgramNo_f = 1) prog 
on tia.GroupFirstAuction_f = prog.GroupFirstAuction_f)
UNION ALL
(select * from Miss)) tia2


left join
LoanInter expX 
on tia2.OwnerTaxRegistrationNumber = expX.PropertyOwnerTaxNumber

left join
expDexp dexp 
on expX.TaxNumber = dexp.ExpTaxNumber 

left join
Exprt
on  dexp.AgreementKey = Exprt.AgreementKey

left join 
`p-cdh`.derivative.dparty party 
on  dexp.ExpPartyKey       =party.PartyKey
and dexp.BusinessDate      =party.BusinessDate
and dexp.ExpSourceSystem   =party.SourceSystem

left join
(select distinct portfolioname, RetailWholesale from `p-cdh`.`primary`.`portfolio` 
where businessdate = (select max(businessdate) from `p-cdh`.`primary`.`portfolio`)) st 
on dexp.PortfolioName = st.PortfolioName

left join
lpsAGR
on  dexp.businessdate = lpsAGR.businessdate   
and dexp.AgreementKey = lpsAGR.AgreementKey and lpsAGR.ranking = 1

left join
crv
on  dexp.AgreementKey = crv.AgreementKey
and dexp.businessdate = crv.businessdate

left join 
sett
on  regexp_replace(regexp_extract(dexp.ExposureKey, '\\d+', 0), '^0+', '') = regexp_replace(regexp_extract(sett.ExposureKey, '\\d+', 0), '^0+', '') --dexp.ExposureKey  = sett.ExposureKey
and dexp.BusinessDate = sett.BusinessDate
and sett.rnk=1

left join
restr
on  dexp.ExposureKey  = restr.NewExposureKey
and dexp.BusinessDate = restr.BusinessDate
and restr.Ranking = 1

left join
OCW
on  dexp.expCDI=OCW.cdi
--and dexp.BusinessDate=OCW.BusinessDate   
and OCW.Ranking=1

left join
L3869
on  dexp.expCDI=L3869.cdi
and dexp.BusinessDate=L3869.BusinessDate 

left join  
prcs_KSX 
on   dexp.AgreementKey = prcs_KSX.AgreementKey 
and  dexp.BusinessDate = prcs_KSX.BusinessDate
and prcs_KSX.rank = 1

left join  
prcs_DP 
on   dexp.AgreementKey = prcs_DP.AgreementKey
and  dexp.BusinessDate = prcs_DP.BusinessDate
and prcs_DP.rank = 1

left join  
prcs_AG 
on   dexp.AgreementKey = prcs_AG.AgreementKey
and  dexp.BusinessDate = prcs_AG.BusinessDate
and  prcs_AG.rank = 1

left join  
prcs_PROS 
on   dexp.AgreementKey = prcs_PROS.AgreementKey
and  dexp.BusinessDate = prcs_PROS.BusinessDate
and  prcs_PROS.rank = 1

left join   
strm
on  dexp.exposurekey=strm.ExposureKey
and dexp.BusinessDate=strm.BusinessDate 

) al
) al2
) ))

In [0]:
%sql
REPAIR TABLE `p-cdh-delivery`.`tia`.auctionmonitoring SYNC METADATA